# SMS Spam Detection — Exploratory Data Analysis

This notebook is for exploration only. Training, evaluation, and inference
live in `src/train.py`, `src/evaluate.py`, and `src/predict.py`.

**Before running**: place `spam.csv` in `../data/` and run from the repo root.

In [ ]:
import sys
sys.path.insert(0, '..')  # so src.* imports work from notebooks/

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

from src.train import load_dataset
from src.preprocess import TextPreprocessor

## 1. Load and inspect data

In [ ]:
df = load_dataset()
print(df.shape)
df.head(10)

In [ ]:
# Class distribution
counts = df['label'].value_counts()
print(counts)
print(f"\nSpam rate: {counts[1] / len(df):.1%}")

In [ ]:
# Class balance bar chart
fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(['ham (0)', 'spam (1)'], [counts.get(0, 0), counts.get(1, 0)],
       color=['steelblue', 'tomato'], edgecolor='grey')
ax.set_ylabel('Count')
ax.set_title('Class Distribution')
plt.tight_layout()
plt.show()

## 2. Message length distribution

In [ ]:
df['msg_len'] = df['text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)
for ax, (label, name) in zip(axes, [(0, 'ham'), (1, 'spam')]):
    subset = df[df['label'] == label]['msg_len']
    ax.hist(subset, bins=40, color='steelblue' if label == 0 else 'tomato',
            edgecolor='grey', alpha=0.8)
    ax.set_title(f'{name} message lengths')
    ax.set_xlabel('Characters')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print(df.groupby('label')['msg_len'].describe())

## 3. Preprocessing preview

In [ ]:
pre = TextPreprocessor()
sample_texts = df.sample(5, random_state=42)['text'].tolist()

for raw in sample_texts:
    cleaned = pre.transform([raw])[0]
    print(f'RAW     : {raw[:80]}')
    print(f'CLEANED : {cleaned[:80]}')
    print()

## 4. Run training (optional)

You can kick off the full training pipeline from here, but it's cleaner
to run it from the terminal:

```bash
python -m src.train
```

In [ ]:
# Uncomment to run training from the notebook:
# from src.train import train
# train()

## 5. Load saved results

In [ ]:
import json
from src.config import METRICS_PATH, MODEL_COMPARISON_CSV

if METRICS_PATH.exists():
    with open(METRICS_PATH) as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2))
else:
    print('metrics.json not found — run training first.')

In [ ]:
if MODEL_COMPARISON_CSV.exists():
    cmp_df = pd.read_csv(MODEL_COMPARISON_CSV)
    display(cmp_df[['pipeline_name', 'cv_f1_mean', 'cv_f1_std']].sort_values('cv_f1_mean', ascending=False))
else:
    print('model_comparison.csv not found — run training first.')

## 6. Live inference (after training)

```python
from src.predict import predict
result = predict("Free entry! Win £1000 now — call 07843")
print(result)
```